In [ ]:
from sqlalchemy import create_engine
from dotenv import load_dotenv
from rasterio.merge import merge
from shapely.geometry import mapping
from shapely import geometry
import os
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from PIL import Image
import json
import sys
# append the path of the parent directory
sys.path.append("..")
from utils.dol import *

load_dotenv()
VRT_PATH = "/app/data/images/2024/aerial/herault/docker_herault_aerial_2024.vrt"
OUTPUT_DIR = "/app/data/datasets/generated/aigle-dol_aerial_v1.1"
ID_COL = "id_ilot"

os.makedirs(OUTPUT_DIR, exist_ok=True)
db_string = os.getenv('DB_STRING_PROD')
engine = create_engine(db_string)

gdf_datas = gpd.read_file('/app/data/datasets/debug/bush/labels/target_dol_zones_v1.gpkg',driver='GPKG')
gdf_datas.to_crs('EPSG:2154', inplace=True)
gdf_datas.head(5)

/opt/conda/lib/python3.11/site-packages/pyogrio/raw.py:198: RuntimeWarning: driver GPKG does not support open option DRIVER
  return ogr_read(


,image_path,id_ilot,code_ilot,compte_communal,insee_com,target_control,target_pv,geometry
0,/app/runs/aigle_aerial_yolov_2024_bedarieux_34...,13591,340028+00046_4,340028+00046,34028,1.0,0.0,"MULTIPOLYGON (((714006.51 6278414.37, 714010.1..."
1,/app/runs/aigle_aerial_yolov_2024_bedarieux_34...,13609,340028+00077_1,340028+00077,34028,1.0,0.0,"MULTIPOLYGON (((712766.502 6278928.377, 712764..."
2,/app/runs/aigle_aerial_yolov_2024_bedarieux_34...,13656,340028+00136_0,340028+00136,34028,1.0,1.0,"MULTIPOLYGON (((715183.909 6281380.426, 715181..."
3,/app/runs/aigle_aerial_yolov_2024_bedarieux_34...,13669,340028+00150_0,340028+00150,34028,1.0,1.0,"MULTIPOLYGON (((712573.269 6277765.338, 712573..."
4,/app/runs/aigle_aerial_yolov_2024_bedarieux_34...,13674,340028+00155_0,340028+00155,34028,1.0,0.0,"MULTIPOLYGON (((716186.71 6278065.79, 716199.1..."


In [3]:
gdf_datas

,image_path,id_ilot,code_ilot,compte_communal,insee_com,target_control,target_pv,geometry
0,/app/runs/aigle_aerial_yolov_2024_bedarieux_34...,13591,340028+00046_4,340028+00046,34028,1.0,0.0,"MULTIPOLYGON (((714006.51 6278414.37, 714010.1..."
1,/app/runs/aigle_aerial_yolov_2024_bedarieux_34...,13609,340028+00077_1,340028+00077,34028,1.0,0.0,"MULTIPOLYGON (((712766.502 6278928.377, 712764..."
2,/app/runs/aigle_aerial_yolov_2024_bedarieux_34...,13656,340028+00136_0,340028+00136,34028,1.0,1.0,"MULTIPOLYGON (((715183.909 6281380.426, 715181..."
3,/app/runs/aigle_aerial_yolov_2024_bedarieux_34...,13669,340028+00150_0,340028+00150,34028,1.0,1.0,"MULTIPOLYGON (((712573.269 6277765.338, 712573..."
4,/app/runs/aigle_aerial_yolov_2024_bedarieux_34...,13674,340028+00155_0,340028+00155,34028,1.0,0.0,"MULTIPOLYGON (((716186.71 6278065.79, 716199.1..."
...,...,...,...,...,...,...,...,...
2129,/app/runs/aigle_aerial_yolov_2024_velieux_3432...,162411,340326S00020_0,340326S00020,34326,0.0,0.0,"MULTIPOLYGON (((678314.851 6254161.039, 678314..."
2130,/app/runs/aigle_aerial_yolov_2024_velieux_3432...,162412,340326S00023_0,340326S00023,34326,0.0,0.0,"MULTIPOLYGON (((678451.568 6254245.819, 678450..."
2131,/app/runs/aigle_aerial_yolov_2024_velieux_3432...,162413,340326T00005_0,340326T00005,34326,1.0,0.0,"MULTIPOLYGON (((678309.252 6255265.524, 678304..."
2132,/app/runs/aigle_aerial_yolov_2024_velieux_3432...,162414,340326V00015_0,340326V00015,34326,1.0,0.0,"MULTIPOLYGON (((678446.094 6253772.391, 678446..."


In [ ]:
"""using a vrt raster  (in crs 2154)available here /path_local_raster
for each row of geodataframe gdf_datas (in crs 2154):
    extract mask of raster using geometry of row
    build a png image with transparency (or alpha?) band for empty values
    store this image to /path_to_dataset
"""
with rasterio.open(VRT_PATH) as src:

    nodata = src.nodata

    for idx, row in gdf_datas.iterrows():
        out_name = f"{row[ID_COL]}.png"
        out_path = os.path.join(OUTPUT_DIR, out_name)
        
        if not os.path.exists(out_path):
            geom = row.geometry
            if geom.area > 200 :
                # --- Mask raster with polygon ---
                out_image, out_transform = mask(
                    src,
                    [geom],
                    crop=True,
                    nodata=nodata,
                    filled=True
                )

                # out_image shape: (bands, height, width)
                bands, height, width = out_image.shape

                # --- Convert to uint8 if needed (example scaling) ---
                if out_image.dtype != np.uint8:
                    out_image = np.clip(out_image, 0, 255).astype(np.uint8)

                # --- Build alpha channel ---
                if nodata is not None:
                    valid_mask = np.any(out_image != nodata, axis=0)
                else:
                    valid_mask = np.any(out_image != 0, axis=0)

                alpha = (valid_mask * 255).astype(np.uint8)

                # --- Prepare RGB ---
                if bands >= 3:
                    rgb = np.stack(
                        [out_image[0], out_image[1], out_image[2]],
                        axis=-1
                    )
                elif bands == 1:
                    rgb = np.repeat(out_image[0][:, :, None], 3, axis=2)
                else:
                    raise ValueError("Unsupported band count")

                # --- Add alpha channel ---
                rgba = np.dstack([rgb, alpha])

                # --- Save PNG ---
                print(f"test image content : max is {np.max(rgba)},min is {np.min(rgba)}, shape is {rgba.shape}")
                print (f"test save path : {out_path}")
                Image.fromarray(rgba, mode="RGBA").save(out_path)

                print(f"Saved: {out_path}")
            else:
                print(f"sample rejected : {row[ID_COL]} area < 200m²")
        else:
            print(f"Detected existing path : {out_path} - skipping row")